In [ ]:
import torch
from torch.utils.data import DataLoader
import lightning as L
from lightning.fabric import seed_everything
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append('..')
from model import OptimHIM
from dataset import SeqTeleopDataset

%load_ext autoreload
%autoreload 2

In [ ]:
seed_everything(42)
batch_size = 1 # Force batch size to 1 for now
max_epochs = 200

In [ ]:
setting = 'task_[0.001 0.001]_policy_[0.05 0.01]'
dataset = SeqTeleopDataset(f'../data/lqr_optimal/{setting}', index=[0, 1])
dataloader = DataLoader(dataset, batch_size=batch_size)

In [ ]:
model = OptimHIM(lr=1e-3)

In [ ]:
logger = TensorBoardLogger(
    save_dir=Path('../logs'),
    name=setting,
)

lr_callback = LearningRateMonitor(
    logging_interval="epoch"
)

In [ ]:
trainer = L.Trainer(
    max_epochs=max_epochs,
    devices=[0],
    logger=logger,
    callbacks=[
        lr_callback
    ],
)

In [ ]:
trainer.fit(model, dataloader)

In [ ]:
model.B.data

In [ ]:
model.B.grad

# Test

In [ ]:
data = next(iter(dataloader))
states, actions, states_next = data
x, x_goal = torch.chunk(states, 2, dim=-1)

In [ ]:
u_H: torch.Tensor = actions.squeeze(0)
u_H_star = model.training_step(data, 0).squeeze(0)

In [ ]:
u_H_star